# 🛡️ Zero-Trust Remote LLM & VLM Inference Gateway

Run an OpenAI-compatible local engine (`llama-server` / `vLLM` / `Ollama` bound strictly to `127.0.0.1:8080`) wrapped behind an **application-layer AES-256-GCM AEAD gateway** on untrusted or shared cloud GPUs (RunPod, Vast.ai, AutoDL, Lambda, or SSH port-forwarded Jupyter pods).

- **Zero Edge Plaintext**: Prompts, system instructions, multimodal images, and streaming SSE responses are encrypted end-to-end between your workstation (`127.0.0.1:18443`) and GPU memory.
- **Dynamic Route & Replay Guard**: Generates a fresh random route (`/v1/ks-<hex>`), random 128-bit session salt, and bounded request-ID anti-replay filter on every launch. Unauthenticated probes receive `403 Forbidden`.


In [ ]:
# === Cell 1: Dependencies & In-Memory Passphrase Unlock ===
import subprocess, sys, os, json, time, secrets, getpass
from collections import deque

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "cryptography", "huggingface_hub"])

from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

KDF_ITERATIONS = 210_000
SALT_BYTES, NONCE_BYTES, TAG_BYTES = 16, 12, 16

def derive_key(pwd: str, salt: bytes) -> bytes:
    kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=KDF_ITERATIONS)
    return kdf.derive(pwd.encode("utf-8"))

def encrypt_payload_hex(data: dict, pwd: str, aad: bytes = b"kissshot-payload-v1") -> str:
    salt, nonce = secrets.token_bytes(SALT_BYTES), secrets.token_bytes(NONCE_BYTES)
    ct = AESGCM(derive_key(pwd, salt)).encrypt(nonce, json.dumps(data, ensure_ascii=False, separators=(",", ":")).encode("utf-8"), aad)
    return (salt + nonce + ct).hex()

def decrypt_payload_hex(hex_str: str, pwd: str, aad: bytes = b"kissshot-payload-v1") -> dict:
    raw = bytes.fromhex("".join(hex_str.split()))
    pt = AESGCM(derive_key(pwd, raw[:SALT_BYTES])).decrypt(raw[SALT_BYTES:SALT_BYTES + NONCE_BYTES], raw[SALT_BYTES + NONCE_BYTES:], aad)
    return json.loads(pt.decode("utf-8"))

def aead_seal(cipher: AESGCM, pt: bytes, aad: bytes) -> bytes:
    nonce = secrets.token_bytes(NONCE_BYTES)
    return nonce + cipher.encrypt(nonce, pt, aad)

def aead_open(cipher: AESGCM, wire: bytes, aad: bytes) -> bytes:
    if len(wire) < NONCE_BYTES + TAG_BYTES:
        raise ValueError("Truncated frame")
    return cipher.decrypt(wire[:NONCE_BYTES], wire[NONCE_BYTES:], aad)

SESSION_PASSPHRASE = getpass.getpass("🔑 Enter your local ZTGB passphrase (in-memory only): ").strip()
assert SESSION_PASSPHRASE, "Passphrase cannot be empty."
print("✅ Passphrase unlocked in memory.")


In [ ]:
# === Cell 2: Start AEAD Gateway in Front of Local Inference Engine (127.0.0.1:8080) ===
# Set INGRESS_HTTPS_URL to your GPU pod's HTTPS proxy URL (e.g. https://<pod>-8081.proxy.runpod.net)
# or http://127.0.0.1:8081 when using SSH local port forwarding (-L 8081:127.0.0.1:8081).
INGRESS_HTTPS_URL = "https://your-pod-8081.example.com"
UPSTREAM_ENGINE_URL = "http://127.0.0.1:8080"
GATEWAY_LISTEN_HOST = "0.0.0.0"
GATEWAY_LISTEN_PORT = 8081

from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import threading, urllib.request, urllib.error

ROUTE_PATH = f"/v1/ks-{secrets.token_hex(8)}"
SESSION_SALT = secrets.token_bytes(SALT_BYTES)
SESSION_CIPHER = AESGCM(derive_key(SESSION_PASSPHRASE, SESSION_SALT))
_SEEN_IDS, _ID_QUEUE, _LOCK = set(), deque(), threading.Lock()

def _check_replay(req_id: str) -> bool:
    if not req_id or len(req_id) < 8:
        return False
    with _LOCK:
        if req_id in _SEEN_IDS:
            return False
        if len(_ID_QUEUE) >= 4096:
            _SEEN_IDS.discard(_ID_QUEUE.popleft())
        _ID_QUEUE.append(req_id)
        _SEEN_IDS.add(req_id)
        return True

class NotebookAEADGateway(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"
    def log_message(self, fmt, *args):
        return
    def _deny(self):
        raw = b'{"error":"forbidden"}'
        self.send_response(403)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(raw)))
        self.send_header("Connection", "close")
        self.end_headers()
        self.wfile.write(raw)
    def do_GET(self):
        self._deny()
    def do_POST(self):
        if self.path != ROUTE_PATH:
            self._deny()
            return
        try:
            n = int(self.headers.get("Content-Length", "0") or "0")
            wire = self.rfile.read(n) if 0 < n <= 64 * 1024 * 1024 else b""
            plain = aead_open(SESSION_CIPHER, wire, f"kissshot-c2s-v1:{ROUTE_PATH}".encode("utf-8"))
            meta_b, req_body = plain.split(b"\n", 1)
            meta = json.loads(meta_b.decode("utf-8"))
            if abs(time.time() - int(meta.get("ts", 0))) > 300 or not _check_replay(str(meta.get("id", ""))): 
                self._deny()
                return
        except Exception:
            self._deny()
            return
        req_id = meta["id"]
        method = meta.get("method", "POST")
        up_req = urllib.request.Request(
            f"{UPSTREAM_ENGINE_URL.rstrip('/')}{meta['path']}",
            data=(req_body if method == "POST" else None),
            headers={"Content-Type": "application/json"} if req_body else {},
            method=method,
        )
        try:
            with urllib.request.urlopen(up_req, timeout=300) as resp:
                ct = (resp.headers.get("Content-Type") or "").lower()
                if "text/event-stream" in ct:
                    self.send_response(200)
                    self.send_header("Content-Type", "text/event-stream; charset=utf-8")
                    self.send_header("Cache-Control", "no-cache")
                    self.send_header("Connection", "close")
                    self.end_headers()
                    seq = 0
                    while True:
                        ln = resp.readline()
                        if not ln:
                            break
                        enc = aead_seal(SESSION_CIPHER, ln, f"kissshot-s2c:{req_id}:{seq}".encode("ascii")).hex()
                        seq += 1
                        self.wfile.write(f"data: {enc}\n\n".encode("ascii"))
                        self.wfile.flush()
                    eof = aead_seal(SESSION_CIPHER, b"", f"kissshot-eof:{req_id}:{seq}".encode("ascii")).hex()
                    self.wfile.write(f"data: EOF:{eof}\n\n".encode("ascii"))
                    self.wfile.flush()
                else:
                    out_b = resp.read()
                    hdr_b = json.dumps({"status": resp.status, "ct": resp.headers.get("Content-Type", "application/json")}).encode("utf-8")
                    enc_all = aead_seal(SESSION_CIPHER, hdr_b + b"\n" + out_b, f"kissshot-s2c-body:{req_id}".encode("ascii"))
                    self.send_response(200)
                    self.send_header("Content-Type", "application/octet-stream")
                    self.send_header("Content-Length", str(len(enc_all)))
                    self.send_header("Connection", "close")
                    self.end_headers()
                    self.wfile.write(enc_all)
        except urllib.error.HTTPError as he:
            err_b = he.read()
            hdr_b = json.dumps({"status": he.code, "ct": "application/json"}).encode("utf-8")
            enc_err = aead_seal(SESSION_CIPHER, hdr_b + b"\n" + err_b, f"kissshot-s2c-body:{req_id}".encode("ascii"))
            self.send_response(200)
            self.send_header("Content-Type", "application/octet-stream")
            self.send_header("Content-Length", str(len(enc_err)))
            self.send_header("Connection", "close")
            self.end_headers()
            self.wfile.write(enc_err)

ticket_payload = {
    "v": 1,
    "type": "kissshot_handshake",
    "url": INGRESS_HTTPS_URL.strip().rstrip("/"),
    "route": ROUTE_PATH,
    "salt": SESSION_SALT.hex(),
    "ts": int(time.time()),
}
ticket_line = "KISSSHOT_TICKET:" + encrypt_payload_hex(ticket_payload, SESSION_PASSPHRASE, aad=b"kissshot-ticket-v1")

gw = ThreadingHTTPServer((GATEWAY_LISTEN_HOST, GATEWAY_LISTEN_PORT), NotebookAEADGateway)
print("=" * 76)
print("🎯 Remote AEAD Gateway Ready! Copy the KISSSHOT_TICKET line below to your workstation:\n")
print(ticket_line)
print("\n" + "=" * 76)
try:
    gw.serve_forever()
except KeyboardInterrupt:
    gw.shutdown()
